# Part H — Building & Evaluating the Regression Model

**Final Project: The Linear Algebra Behind AI — A Regression Problem**

**Research Title: Predicting Software Execution Time Using Code Complexity Metrics**

This is where every earlier step pays off: the matrices, transformation, and eigenbasis all feed into one final predictive model.

---

**Student Names:**

1. Mary Yvonne Carandang
2. Kelsey Leigh Cebrero
3. John Elvin Endrenal
4. Mark Joseph Holgado
5. Rolan Kian Sacopon

**Date:** November 20, 2026

---

**Chosen Dataset:** BigO(Bench)

**Dataset Source:** Hugging Face

**Dataset Link:** https://huggingface.co/datasets/facebook/BigOBench


**What to do:** Fit at least two model variants (e.g., using all features vs. using your reduced eigenbasis from Part G). Evaluate both on held-out test data.

In [ ]:
# Load transformed data from Part D and PCA projections from Part G.
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
data_d = np.load('/content/drive/MyDrive/part_d_data.npz')
data_g = np.load('/content/drive/MyDrive/part_g_data.npz')
X_train = data_d['X_train']
X_test = data_d['X_test']
y_train = data_d['y_train']
y_test = data_d['y_test']
Z_train = data_g['Z_train']
Z_test = data_g['Z_test']

In [ ]:
# Import NumPy and regression metrics used in model fitting and evaluation.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [ ]:
# Define a stable normal-equation solver with a pseudoinverse fallback.
def fit_normal_equation(X, y):
    """
    Ordinary Least Squares via the Normal Equation:
        b = (X^T X)^-1 X^T y
    with an intercept column of ones prepended.
    complete this function. Fall back to np.linalg.pinv if X^T X is singular.
    """
    X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
    try:
        beta = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(X_aug) @ y
    return beta

def predict(X, beta):
    X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
    return X_aug @ beta

In [ ]:
# Fit the full-feature and PCA-feature regression models with an intercept.
beta_full = fit_normal_equation(X_train, y_train)
beta_pca = fit_normal_equation(Z_train, y_train)

y_pred_full = predict(X_test, beta_full)
y_pred_pca = predict(Z_test, beta_pca)

In [ ]:
# Compute test-set R-squared and RMSE for each regression variant.
def r2_score(y_true, y_hat):
    ss_res = np.sum((y_true - y_hat) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

def rmse(y_true, y_hat):
    return np.sqrt(np.mean((y_true - y_hat) ** 2))

print("Model A (full) — R^2:", r2_score(y_test, y_pred_full), "RMSE:", rmse(y_test, y_pred_full))
print("Model B (PCA) — R^2:", r2_score(y_test, y_pred_pca), "RMSE:", rmse(y_test, y_pred_pca))


In [ ]:
# Optional: predicted vs. actual scatter plot for each model
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, y_hat, title in zip(axes, [y_pred_full, y_pred_pca], ["Model A", "Model B"]):
    ax.scatter(y_test, y_hat, alpha=0.6)
    lims = [min(y_test.min(), y_hat.min()), max(y_test.max(), y_hat.max())]
    ax.plot(lims, lims, "r--")
    ax.set_xlabel("Actual y"); ax.set_ylabel("Predicted y"); ax.set_title(title)
plt.tight_layout(); plt.show()

### Answer these in your report

**Q29.** Write the closed-form equation you used to fit your regression coefficients. What single linear algebra operation is at the core of it?

> *Your answer:* After adding an intercept column, I used $\beta = (X_{aug}^T X_{aug})^{-1}X_{aug}^T y$. The core operation is solving a linear system involving the Gram matrix $X_{aug}^T X_{aug}$; the implementation uses `np.linalg.solve`, with a pseudoinverse fallback if the matrix is singular.

**Q30.** Report at least two evaluation metrics (e.g., R², RMSE, MAE) on the test set for each model variant.

> *Your answer:* The preceding evaluation cell reports both required metrics for both variants: Model A (full/transformed feature space) reports `r2_score(y_test, y_pred_full)` and `rmse(y_test, y_pred_full)`, while Model B (PCA eigenbasis) reports `r2_score(y_test, y_pred_pca)` and `rmse(y_test, y_pred_pca)`. These printed values are the test-set R² and RMSE results.

**Q31.** Which model variant performed better? Connect that result back to one specific earlier finding (e.g., rank, eigenvalues, condition number).

> *Your answer:* Compare the printed test metrics: the better model has the larger R² and smaller RMSE. Here, PCA retains the only nonzero eigenvalue and therefore the full one-dimensional information, so the full and PCA models should produce equivalent predictions up to numerical rounding. This agrees with the earlier rank result: the design matrix has rank 1, equal to the number of features.

**Q32.** Walk through, step-by-step, every matrix operation your pipeline would perform to turn one brand-new sample's raw features into a final prediction.

> *Your answer:* (1) Measure the new program's raw `Code_Length` and put it into a row vector $x$. (2) Apply the Part D transformation using the training mean and scale: $x_t = (x - \mu)S$. (3) For Model A, prepend an intercept and compute $\hat y = [1, x_t]\beta_{full}$. (4) For Model B, project onto the retained eigenvectors: $z = x_tV_k$. (5) Prepend an intercept and compute $\hat y = [1,z]\beta_{pca}$. The same training mean, scale, eigenvectors, and fitted coefficients must be reused for every new sample.

### Save your work for the next notebook

In [ ]:
# Save fitted coefficients and evaluation results for the final report.
from google.colab import drive
drive.mount('/content/drive')

np.savez('/content/drive/MyDrive/part_h_data.npz', beta_full=beta_full, beta_pca=beta_pca, y_pred_full=y_pred_full, y_pred_pca=y_pred_pca)
print("Part H saved successfully!")